# 03 — Drug-class patterns and time

This notebook compares overall/NRTI/NNRTI/PI transmitted drug resistance and explores the time distribution of the source studies. It is designed to show why a single timeless global map can be misleading.

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA = Path("data")
summary_path = DATA / "hivdb_country_summary_2015.csv"
studies_path = DATA / "hivdb_surveillance_studies_2015.csv"
study_countries_path = DATA / "hivdb_study_countries_2015.csv"
metadata_path = DATA / "hivdb_2015_metadata.json"

required = [summary_path, studies_path, study_countries_path, metadata_path]
missing = [p.name for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Generated data are missing: " + ", ".join(missing) +
        ". In the deployed GitHub Pages site these files are generated during the build. "
        "For local use, run: python scripts/fetch_plos_2015.py --output-dir content/data"
    )

summary = pd.read_csv(summary_path)
studies = pd.read_csv(studies_path)
study_countries = pd.read_csv(study_countries_path)
metadata = json.loads(metadata_path.read_text())


In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px

outcomes = [c for c in ["tdr_overall_pct", "tdr_nrti_pct", "tdr_nnrti_pct", "tdr_pi_pct"] if c in summary.columns]
summary[outcomes].describe().T

## Compare drug classes for countries with enough evidence

The threshold below is pedagogical rather than inferential. Change it and observe how the set of mapped countries changes.

In [ ]:
min_studies = 2
eligible = summary.loc[summary["n_studies_single_country"] >= min_studies].copy()
long = eligible.melt(
    id_vars=["country", "iso3", "n_studies_single_country", "n_participants_weighted"],
    value_vars=outcomes,
    var_name="outcome",
    value_name="tdr_pct",
).dropna(subset=["tdr_pct"])
long["outcome"] = long["outcome"].str.replace("tdr_", "", regex=False).str.replace("_pct", "", regex=False).str.upper()
long.sort_values(["country", "outcome"]).head(20)

In [ ]:
fig = px.box(
    long,
    x="outcome",
    y="tdr_pct",
    points="all",
    hover_name="country",
    labels={"tdr_pct": "Weighted TDR (%)", "outcome": "Drug-resistance outcome"},
    title=f"Country-level descriptive TDR values (countries with ≥{min_studies} eligible studies)",
)
fig.show()

## Study years by region/country

The source spreadsheet provides median sample year. Here we inspect the newest and oldest study medians represented for each single-country study association.

In [ ]:
single = study_countries.loc[study_countries["is_single_country"].astype(bool)].copy()
years = (
    single.dropna(subset=["iso3", "median_sample_year"])
    .groupby(["iso3", "country"], as_index=False)
    .agg(first_median_year=("median_sample_year", "min"),
         last_median_year=("median_sample_year", "max"),
         n_studies=("study_id", "nunique"))
)
years.sort_values("last_median_year").head(20)

In [ ]:
fig = px.choropleth(
    years,
    locations="iso3",
    color="last_median_year",
    hover_name="country",
    hover_data=["first_median_year", "last_median_year", "n_studies"],
    color_continuous_scale="Viridis",
    title="Most recent median sample year represented in the 2015 source snapshot",
)
fig.update_geos(showframe=False, showcoastlines=True, projection_type="natural earth")
fig.update_layout(margin=dict(l=0, r=0, t=60, b=0))
fig.show()

## Extend this notebook with current HIVDB data

When a stable current HIVDB surveillance export is available, normalize it with `scripts/fetch_current_hivdb.py`, keep its retrieval metadata, and rerun these same cells against the current country summary. Keeping identical analysis code while swapping a versioned data snapshot makes temporal comparisons easier to audit.